# Optimization for a New Patient

Use this notebook to run optimization for a patient not in the holdout file.

You can input patient data in either way:
- **Dict mode (single patient, quickest):** edit the Python dictionary in Cell 5.
- **CSV mode (multiple patients):** edit one or more rows in `data/raw/new_patient_input_template.csv`, then set `INPUT_MODE = "csv"` in Cell 5.

The notebook will show an input validation review before running optimization:
- **OK** = accepted
- **ERROR** = invalid or missing value; fix before running

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import importlib
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

from src import config
import src.optimization_utils as ou
import src.display as disp
import src.runs as runs

importlib.reload(config)
importlib.reload(ou)
importlib.reload(disp)
importlib.reload(runs)

pd.set_option("display.max_columns", None)

In [2]:
# GA configuration
POP_SIZE = 100
N_GEN = 10
SEED = 42
TOP_N = 4
SCORE_TOL = 5

PSO_LL_OVERRIDE = False
DEDUP_BEST = True

SCENARIOS = [
    "equal",
    "mech_fail",
    "mech_fail_t4l1pa",
    "l4s1",
    "t4l1pa",
    "gap_score",
    "gap_score_mech_fail",
    "ll",
    "odi",
]

for key in SCENARIOS:
    runs.print_preset(runs.PRESETS[key])


  Equal Weights (Composite)
  All 6 alignment components and mechanical failure weighted equally
                       GAP Score: 0.1429 ◀
                    L1PA penalty: 0.1429 ◀
                    L4S1 penalty: 0.1429 ◀
                  T4L1PA penalty: 0.1429 ◀
                      LL penalty: 0.1429 ◀
        GAP category improvement: 0.1429 ◀
         Mechanical failure prob: 0.1429 ◀

  Minimize Mechanical Failure
  Primarily mechanical failure probability, with small alignment weights
                       GAP Score: 0.0500 ◀
                    L1PA penalty: 0.0500 ◀
                    L4S1 penalty: 0.0500 ◀
                  T4L1PA penalty: 0.0500 ◀
                      LL penalty: 0.0500 ◀
        GAP category improvement: 0.0500 ◀
         Mechanical failure prob: 0.7000 ◀

  Minimize Mechanical Failure + T4L1PA
  Mechanical failure probability and T4PA-L1PA mismatch weighted equally
                       GAP Score: 0.0000
                    L1PA penalty: 0.0000
 

In [3]:
# Load model bundles
model_configs = {
    "Mechanical Failure": config.MECH_FAIL_MODEL,
    "L4S1": config.L4S1_MODEL,
    "LL": config.LL_MODEL,
    "T4PA": config.T4PA_MODEL,
    "L1PA": config.L1PA_MODEL,
    "SVA": config.SVA_MODEL,
    "SS": config.SS_MODEL,
    "Global Tilt": config.GLOBAL_TILT_MODEL,
    "ODI": config.ODI_MODEL,
}

bundles = {name: ou.load_model_bundle(path) for name, path in model_configs.items()}

mech_fail_bundle = bundles["Mechanical Failure"]
ODI_bundle = bundles["ODI"]

delta_bundles = {
    "L4S1": bundles["L4S1"],
    "LL": bundles["LL"],
    "T4PA": bundles["T4PA"],
    "L1PA": bundles["L1PA"],
    "SS": bundles["SS"],
    "GlobalTilt": bundles["Global Tilt"],
    "SVA": bundles["SVA"],
}

UIV_CHOICES, xl, xu = ou.get_decision_config()
print("Models loaded.")

Models loaded.


In [4]:
# Input mode: "dict" or "csv"
INPUT_MODE = "dict"
CSV_PATH = Path.cwd().parent / "data" / "raw" / "new_patient_input_template.csv"

# Default example: holdout patient 1176294
patient_input = {
    "id": 1176294,
    "age": 71,
    "sex": "MALE",
    "bmi": 28.02,
    "C7CSVL_preop": 26.2,
    "SVA_preop": 83.6,
    "T4PA_preop": 27.6,
    "L1PA_preop": 18.7,
    "LL_preop": 22.7,
    "L4S1_preop": 15.8,
    "PT_preop": 33.1,
    "PI_preop": 50.4,
    "SS_preop": 17.3,
    "cobb_main_curve_preop": 21.5,
    "FC_preop": 17.4,
    "tscore_femneck_preop": -1.5,
    "HU_UIV_preop": 78.0,
    "HU_UIVplus1_preop": 87.0,
    "HU_UIVplus2_preop": 98.0,
    "global_tilt_preop": 40.3,
    "CCI": 2,
    "ASA_CLASS": 3,
    "ODI_preop": 42.0,
    "revision": 1,
    "smoking": 1,
}

if INPUT_MODE == "csv":
    input_preview_df = pd.read_csv(CSV_PATH)
    if len(input_preview_df) == 0:
        raise ValueError(f"CSV file has no rows: {CSV_PATH}")
    print(f"Loaded {len(input_preview_df)} patient(s) from CSV: {CSV_PATH}")
elif INPUT_MODE == "dict":
    input_preview_df = pd.DataFrame([patient_input])
    print("Using patient_input dictionary")
else:
    raise ValueError("INPUT_MODE must be 'dict' or 'csv'")

input_preview_df

Using patient_input dictionary


,id,age,sex,bmi,C7CSVL_preop,SVA_preop,T4PA_preop,L1PA_preop,LL_preop,L4S1_preop,PT_preop,PI_preop,SS_preop,cobb_main_curve_preop,FC_preop,tscore_femneck_preop,HU_UIV_preop,HU_UIVplus1_preop,HU_UIVplus2_preop,global_tilt_preop,CCI,ASA_CLASS,ODI_preop,revision,smoking
0,1176294,71,MALE,28.02,26.2,83.6,27.6,18.7,22.7,15.8,33.1,50.4,17.3,21.5,17.4,-1.5,78.0,87.0,98.0,40.3,2,3,42.0,1,1


In [5]:
review_df = ou.review_new_patient_inputs(
    input_mode=INPUT_MODE,
    patient_input=patient_input,
    csv_path=CSV_PATH,
)

display(Markdown("### Input Validation Review"))
display(review_df)

error_count = (review_df["Status"] == "ERROR").sum()
print(f"Validation summary: {error_count} error(s)")

if error_count > 0:
    raise ValueError("Fix input errors shown above before running optimization.")

prepared_patients = ou.prepare_new_patient_inputs(
    input_mode=INPUT_MODE,
    patient_input=patient_input,
    csv_path=CSV_PATH,
)

print(f"Prepared {len(prepared_patients)} patient(s) for optimization.")

for i, rec in enumerate(prepared_patients, start=1):
    patient_id = rec["patient_id"]
    patient_fixed = rec["patient_fixed"]
    profile_row = rec["profile_row"]
    gap_score_preop = rec["gap_score_preop"]
    gap_category_preop = rec["gap_category_preop"]

    display(Markdown(f"### Input Profile {i}: {patient_id}"))
    disp.display_patient_preop(patient_fixed, patient_id=patient_id, holdout_row=profile_row)
    print(f"Computed preop GAP: {gap_score_preop} ({gap_category_preop})")

### Input Validation Review

,Patient,Field,Value,Expected,Status,Note
0,1176294,age,71,Numeric,OK,
1,1176294,sex,MALE,MALE/FEMALE (or M/F),OK,
2,1176294,bmi,28.02,Numeric,OK,
3,1176294,C7CSVL_preop,26.2,Numeric,OK,
4,1176294,SVA_preop,83.6,Numeric,OK,
5,1176294,T4PA_preop,27.6,Numeric,OK,
6,1176294,L1PA_preop,18.7,Numeric,OK,
7,1176294,LL_preop,22.7,Numeric,OK,
8,1176294,L4S1_preop,15.8,Numeric,OK,
9,1176294,PT_preop,33.1,Numeric,OK,


Validation summary: 0 error(s)
Prepared 1 patient(s) for optimization.


### Input Profile 1: 1176294

**Patient 1176294 — Preop Profile**

Parameter,Value
DEMOGRAPHICS,
Age,71
Sex,MALE
BMI,28.0
ASA Class,3
CCI,2
Revision,Yes
Smoking,–
────────────,──────────
ALIGNMENT,


Computed preop GAP: 10 (SD)


In [6]:
results_by_patient = {}

for i, rec in enumerate(prepared_patients, start=1):
    patient_id = rec["patient_id"]
    patient_fixed = rec["patient_fixed"]

    display(Markdown(f"---\n## Patient {i}: {patient_id}"))

    all_results = {}
    for key in SCENARIOS:
        preset = runs.PRESETS[key]
        print(f"\n▶ Running: {preset['label']}...")

        result = runs.run_optimization(
            preset=preset,
            patient_fixed=patient_fixed,
            delta_bundles=delta_bundles,
            mech_fail_bundle=mech_fail_bundle,
            xl=xl,
            xu=xu,
            odi_bundle=ODI_bundle,
            pop_size=POP_SIZE,
            n_gen=N_GEN,
            seed=SEED,
            verbose=False,
            top_n=TOP_N,
            score_tolerance=SCORE_TOL,
            pso_ll_override=PSO_LL_OVERRIDE,
        )
        if result is not None:
            all_results[key] = result

    results_by_patient[patient_id] = all_results

    display(Markdown("### Best Solution per Scenario"))
    disp.display_best_per_scenario(
        all_results=all_results,
        patient_fixed=patient_fixed,
        actual_eval=None,
        scenarios=SCENARIOS,
        deduplicate=DEDUP_BEST,
        include_actual=False,
    )

---
## Patient 1: 1176294


▶ Running: Equal Weights (Composite)...

▶ Running: Minimize Mechanical Failure...

▶ Running: Minimize Mechanical Failure + T4L1PA...

▶ Running: Minimize L4S1 Penalty...

▶ Running: Minimize T4L1PA Penalty...

▶ Running: Minimize GAP Score...

▶ Running: Minimize GAP Score and Mechanical Failure...

▶ Running: Minimize LL (PI-LL) Penalty...

▶ Running: Minimize Postop ODI...


### Best Solution per Scenario

Parameter,Equal Weights (Composite),Minimize Mechanical Failure,Minimize Mechanical Failure + T4L1PA,Minimize L4S1 Penalty,Minimize T4L1PA Penalty,Minimize GAP Score,Minimize GAP Score and Mechanical Failure,Minimize LL (PI-LL) Penalty,Minimize Postop ODI
Alignment Score,1.5,1.5,12.8,12.9,12.8,1.6,1.5,12.8,9.2
Optimization Score,7.3,30.0,20.6,8.2,8.2,12.4,24.1,9.0,8.7
Mech Fail Prob,42.1%,42.2%,41.2%,41.1%,41.2%,42.7%,42.1%,41.1%,43.3%
Predicted ODI,19.1,10.9,6.9,12.6,4.5,11.6,13.4,6.9,-0.8
GAP Score,10 (SD) → 1 (P),10 (SD) → 1 (P),10 (SD) → 6 (MD),10 (SD) → 6 (MD),10 (SD) → 6 (MD),10 (SD) → 1 (P),10 (SD) → 1 (P),10 (SD) → 6 (MD),10 (SD) → 3 (MD)
────────────,──────────,──────────,──────────,──────────,──────────,──────────,──────────,──────────,──────────
SURGICAL PLAN,,,,,,,,,
UIV_implant,Hook,Hook,FS,FS,FS,Hook,Hook,FS,FS
num_levels_cat,higher,lower,higher,higher,lower,higher,higher,higher,lower
num_interbody_fusion_levels,5,5,5,5,5,4,5,5,2


In [8]:
SHOW_TOP_N = True

if SHOW_TOP_N:
    for patient_id, all_results in results_by_patient.items():
        display(Markdown(f"## Top {TOP_N} Solutions — {patient_id}"))
        for key in SCENARIOS:
            if key not in all_results:
                continue
            result = all_results[key]
            display(Markdown(f"### {result['label']}"))
            display(result["diverse_df"].head(TOP_N))

## Top 4 Solutions — 1176294

### Equal Weights (Composite)

,UIV_implant,num_levels_cat,num_interbody_fusion_levels,ALIF,XLIF,TLIF,num_rods,num_pelvic_screws,osteotomy,composite_score,optimization_score,mech_fail_prob,gap_score,gap_category,LL_postop,SS_postop,L4S1_postop,GlobalTilt_postop,T4PA_postop,L1PA_postop,SVA_postop,ODI_postop,PI-LL_postop
0,Hook,higher,5,0,0,0,4,2,1,1.48,7.29,42.1%,1,P,46.4,31.9,35.9,18.9,2.6,4.1,82.4,19.1,4.0 ✓
1,Hook,higher,5,0,1,1,4,2,1,1.47,7.29,42.2%,1,P,46.5,31.8,36.2,18.9,5.1,5.9,88.0,13.4,3.9 ✓
2,Hook,lower,5,0,0,0,4,2,1,1.47,7.30,42.2%,1,P,46.4,31.8,35.7,19.0,1.5,3.0,82.7,16.6,4.0 ✓
3,Hook,higher,5,0,1,0,4,2,1,1.52,7.32,42.1%,1,P,46.5,31.8,32.3,18.9,3.4,4.8,88.0,13.4,3.9 ✓


### Minimize Mechanical Failure

,UIV_implant,num_levels_cat,num_interbody_fusion_levels,ALIF,XLIF,TLIF,num_rods,num_pelvic_screws,osteotomy,composite_score,optimization_score,mech_fail_prob,gap_score,gap_category,LL_postop,SS_postop,L4S1_postop,GlobalTilt_postop,T4PA_postop,L1PA_postop,SVA_postop,ODI_postop,PI-LL_postop
0,Hook,higher,5,0,1,0,4,2,1,1.52,29.93,42.1%,1,P,46.5,31.8,32.3,18.9,3.4,4.8,88.0,13.4,3.9 ✓
1,Hook,higher,5,0,0,0,4,2,1,1.48,29.93,42.1%,1,P,46.4,31.9,35.9,18.9,2.6,4.1,82.4,19.1,4.0 ✓
2,Hook,lower,5,0,1,0,4,2,1,1.52,30.00,42.2%,1,P,46.5,31.7,32.1,19.0,2.3,3.7,88.3,10.9,3.9 ✓
3,Hook,lower,5,0,0,0,4,2,1,1.47,30.00,42.2%,1,P,46.4,31.8,35.7,19.0,1.5,3.0,82.7,16.6,4.0 ✓


...

,UIV_implant,num_levels_cat,num_interbody_fusion_levels,ALIF,XLIF,TLIF,num_rods,num_pelvic_screws,osteotomy,composite_score,optimization_score,mech_fail_prob,gap_score,gap_category,LL_postop,SS_postop,L4S1_postop,GlobalTilt_postop,T4PA_postop,L1PA_postop,SVA_postop,ODI_postop,PI-LL_postop
0,Hook,higher,5,0,1,0,4,2,1,1.52,24.13,42.1%,1,P,46.5,31.8,32.3,18.9,3.4,4.8,88.0,13.4,3.9 ✓
1,Hook,higher,5,0,0,0,4,2,1,1.48,24.14,42.1%,1,P,46.4,31.9,35.9,18.9,2.6,4.1,82.4,19.1,4.0 ✓
2,Hook,lower,5,0,1,0,4,2,1,1.52,24.18,42.2%,1,P,46.5,31.7,32.1,19.0,2.3,3.7,88.3,10.9,3.9 ✓
3,Hook,lower,5,0,0,0,4,2,1,1.47,24.19,42.2%,1,P,46.4,31.8,35.7,19.0,1.5,3.0,82.7,16.6,4.0 ✓


### Minimize LL (PI-LL) Penalty

,UIV_implant,num_levels_cat,num_interbody_fusion_levels,ALIF,XLIF,TLIF,num_rods,num_pelvic_screws,osteotomy,composite_score,optimization_score,mech_fail_prob,gap_score,gap_category,LL_postop,SS_postop,L4S1_postop,GlobalTilt_postop,T4PA_postop,L1PA_postop,SVA_postop,ODI_postop,PI-LL_postop
0,FS,higher,5,1,1,0,4,2,1,12.85,8.97,41.1%,6,MD,46.9,31.5,40.4,18.6,3.1,3.0,87.6,6.9,3.5 ✓
1,FS,lower,5,1,1,0,4,2,1,12.85,8.98,41.2%,6,MD,47.0,31.3,40.2,18.7,2.1,1.9,87.9,4.5,3.4 ✓
2,FS,higher,5,1,1,1,4,2,1,12.85,8.99,41.2%,6,MD,46.9,31.5,44.2,18.6,4.8,4.1,87.6,6.9,3.5 ✓
3,FS,lower,5,1,1,1,4,2,1,12.85,9.00,41.3%,6,MD,47.0,31.3,44.0,18.7,3.8,3.0,87.9,4.5,3.4 ✓


### Minimize Postop ODI

,UIV_implant,num_levels_cat,num_interbody_fusion_levels,ALIF,XLIF,TLIF,num_rods,num_pelvic_screws,osteotomy,composite_score,optimization_score,mech_fail_prob,gap_score,gap_category,LL_postop,SS_postop,L4S1_postop,GlobalTilt_postop,T4PA_postop,L1PA_postop,SVA_postop,ODI_postop,PI-LL_postop
0,FS,lower,2,1,1,0,4,2,1,9.04,8.61,43.1%,3,MD,46.5,31.3,36.2,18.8,5.7,4.2,78.2,-0.8,3.9 ✓
1,FS,lower,2,0,1,0,4,2,1,9.47,8.64,43.2%,3,MD,46.1,30.8,27.3,18.8,6.5,6.0,78.5,-0.8,4.3 ✓
2,FS,lower,2,0,1,1,4,2,1,9.17,8.67,43.3%,3,MD,46.1,30.8,31.1,18.8,8.2,7.1,78.5,-0.8,4.3 ✓
3,FS,higher,1,0,1,0,4,2,1,9.65,8.75,43.8%,3,MD,45.3,31.0,26.1,18.8,8.8,7.9,77.8,-0.2,5.1 ✓
